<a href="https://colab.research.google.com/github/lizagabriela80-afk/Data-analyse/blob/main/SQL_sur_des_ventes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import duckdb

In [2]:
rng = np.random.default_rng(42)

In [3]:
n = 500

In [4]:
commandes = pd.DataFrame({
    "id_commande": range(1, n + 1),
    "id_client": rng.integers(1, 80, n),
    "date_commande": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
    "montant": rng.gamma(2, 25, n).round(2),
    "categorie": rng.choice(["Mode", "Maison", "Électronique", "Beauté"], n),
})

In [5]:
con = duckdb.connect()

In [6]:
con.register("commandes", commandes)

Question 1 — CA et nombre de commandes par mois

In [7]:
q1 = """
SELECT
    strftime(date_commande, '%Y-%m')      AS mois,
    COUNT(*)                              AS nb_commandes,
    ROUND(SUM(montant), 2)                AS chiffre_affaires
FROM commandes
GROUP BY mois
ORDER BY mois;
"""
con.sql(q1).df()

,mois,nb_commandes,chiffre_affaires
0,2025-01,36,1733.51
1,2025-02,55,3071.17
2,2025-03,36,1533.15
3,2025-04,38,1899.50
4,2025-05,26,1144.09
5,2025-06,44,2022.87
6,2025-07,47,2073.76
7,2025-08,49,2400.69
8,2025-09,40,2005.15
9,2025-10,43,2215.50


Question 2 — Top 5 clients par montant total


In [8]:
q2 = """
SELECT
    id_client,
    COUNT(*)                    AS nb_commandes,
    ROUND(SUM(montant), 2)      AS montant_total
FROM commandes
GROUP BY id_client
ORDER BY montant_total DESC
LIMIT 5;
"""
con.sql(q2).df()

,id_client,nb_commandes,montant_total
0,35,14,662.88
1,44,11,641.95
2,53,12,589.70
3,13,6,558.40
4,61,10,553.37


Question 3 — Panier moyen par catégorie

In [9]:
q3 = """
SELECT
    categorie,
    COUNT(*)                     AS nb_commandes,
    ROUND(AVG(montant), 2)       AS panier_moyen
FROM commandes
GROUP BY categorie
ORDER BY panier_moyen DESC;
"""
con.sql(q3).df()

,categorie,nb_commandes,panier_moyen
0,Maison,123,50.02
1,Électronique,108,49.23
2,Mode,136,48.16
3,Beauté,133,46.66


Question 4 — Part du CA par catégorie (en %)

In [10]:
q4 = """
SELECT
    categorie,
    ROUND(SUM(montant), 2)                                   AS chiffre_affaires,
    ROUND(100.0 * SUM(montant) / SUM(SUM(montant)) OVER (), 2) AS part_pct
FROM commandes
GROUP BY categorie
ORDER BY part_pct DESC;
"""
con.sql(q4).df()

,categorie,chiffre_affaires,part_pct
0,Mode,6549.20,27.04
1,Beauté,6206.43,25.62
2,Maison,6152.03,25.40
3,Électronique,5316.39,21.95


Conclusion : La répartition du CA par catégorie est très équilibrée (22–26 %), aucune catégorie ne pèse plus d'un quart du total — le  portefeuille est diversifié et peu risqué côté dépendance produit.

